# Senior Project Final Report (mlreport)


# API Imports

`Report` creates a report for a single model. `ComparisonReport` combines two or more built reports into one model-comparison report.


In [ ]:
# If running in Google Colab, uncomment this install cell.
# !rm -rf /content/mlreport
# !git clone https://github.com/Lucas-Summers/mlreport.git
# !pip install /content/mlreport


In [ ]:
from mlreport import ComparisonReport, Report


# API Flow

Most reports follow the same sequence:

1. Create a `Report` around a fitted model.
2. Add evaluation data with `add_split()` or `add_crossval()`.
3. Optionally attach hyperparameter search results with `add_search()`.
4. Call `build()` to compute metrics and plots.
5. Export the report to text, HTML, Markdown, JSON, or PDF.


## 1. Create a Report

For scikit-learn models, `mlreport` usually detects whether the model is for classification or regression. For custom models, pass `model_type` and `model_params` explicitly.

```python
report = Report(
    model,  # fitted sklearn-compatible or custom model
    title="Model Report",
    author="Your Name",
    description="Optional description of this model or experiment.",
    theme="light",  # or "dark"
    cmap="viridis",  # any matplotlib colormap
    model_type=None,  # optional: "classification" or "regression"
    model_params=None,  # optional dict of parameters to display
)
```


## 2. Add Evaluation Data

Choose one evaluation style. A single report should use either named train/test-style splits or cross-validation, not both.

### Option A: Train/Test Splits

```python
report.add_split("train", X_train, y_train)
report.add_split("test", X_test, y_test)

# If predictions are already computed:
report.add_split("test", X_test, y_test, y_pred_test)
```

### Option B: Cross-Validation

```python
report.add_crossval(X, y, cv=5)
report.add_crossval(X, y, cv=cv)
report.add_crossval(X, y, y_pred=y_pred_cv, cv=cv)
report.add_crossval(X, y, y_pred=y_pred_cv, fold_ids=fold_ids)
```


## 3. Add Hyperparameter Search Results

If you used `GridSearchCV` or `RandomizedSearchCV`, attach the fitted search object before building the report.

```python
report.add_search(search_cv)
```

The current tuning plots support exactly two tuned hyperparameters.


## 4. Build Metrics and Plots

```python
report.available_metrics()
report.available_plots()

report.build(
    exclude_metrics=[],
    exclude_plots=[],
)
```


## 5. Export the Report

```python
report.to_txt()
report.to_html("report.html")
report.to_md("report.md")
report.to_json("report.json")
report.to_pdf("report.pdf")
```


## 6. Compare Models

Build individual reports first, then pass them to `ComparisonReport`.

```python
baseline_report = (
    Report(model_a, title="Baseline")
    .add_split("test", X_test, y_test)
    .build()
)

candidate_report = (
    Report(model_b, title="Candidate")
    .add_split("test", X_test, y_test)
    .build()
)

comparison = ComparisonReport(
    reports=[baseline_report, candidate_report],
    title="Model Comparison",
    split="test",
    theme="light",
).build()

comparison.to_pdf("comparison.pdf")
```


# API Reference


## `mlreport.Report`

```python
class mlreport.Report(
    model,
    title: str | None = None,
    author: str | None = None,
    description: str | None = None,
    theme: str = "light",
    cmap: str = "viridis",
    model_type: str | None = None,
    model_params: dict | None = None,
)
```

Build and render an evaluation report for one fitted model or for precomputed
predictions from one model.

`Report` supports two mutually exclusive evaluation-data flows:

- Add named data splits with [`add_split`](#reportadd_split).
- Add one cross-validation prediction set with [`add_crossval`](#reportadd_crossval).

After adding evaluation data, call [`build`](#reportbuild) before exporting or
calling [`to_dict`](#reportto_dict).

### Parameters

`model` : object

- Fitted scikit-learn-compatible estimator or custom model object. If
  predictions are not passed explicitly to `add_split` or `add_crossval`, the
  model must expose a callable `predict(X)` method.

`title` : `str | None`, default=`None`

- Optional report title shown in rendered output.

`author` : `str | None`, default=`None`

- Optional report author shown in rendered output.

`description` : `str | None`, default=`None`

- Optional prose description shown in rendered output and in comparison cards.

`theme` : `str`, default=`"light"`

- Render theme name. Built-in themes include `"light"` and `"dark"`.

`cmap` : `str`, default=`"viridis"`

- Matplotlib colormap name used for generated plots.

`model_type` : `str | None`, default=`None`

- Explicit model type for custom models. Accepted explicit values are
  `"classification"`, `"classifier"`, `"regression"`, and `"regressor"`. If
  omitted, scikit-learn type checks are used.

`model_params` : `dict | None`, default=`None`

- Optional parameter mapping to display in the report. If omitted,
  `model.get_params()` is used when available. Custom models without
  `get_params()` should pass this argument.

### Attributes

`model` : object

- Model object passed to the constructor.

`model_type` : `str | None`

- Explicit model type label, if supplied.

`model_params` : `dict | None`

- Explicit model parameter mapping, if supplied.

`title` : `str | None`

- Report title metadata.

`author` : `str | None`

- Report author metadata.

`description` : `str | None`

- Report description metadata.

`theme` : `str`

- Theme used by text, HTML, Markdown, and PDF renderers.

`cmap` : `str`

- Colormap used for plots.

### Methods

#### `Report.available_metrics`

```python
Report.available_metrics() -> None
```

Print the metric IDs and display names available for the report's detected model
type. The method prints to standard output and does not return a value.

Use these IDs in `build(exclude_metrics=[...])`.

Classification metric IDs:

- `accuracy` : Accuracy
- `precision_macro` : Precision (Macro)
- `recall_macro` : Recall (Macro)
- `f1_macro` : F1 Score (Macro)
- `precision_weighted` : Precision (Weighted)
- `recall_weighted` : Recall (Weighted)
- `f1_weighted` : F1 Score (Weighted)

Regression metric IDs:

- `r2` : R2 Score
- `mse` : Mean Squared Error
- `rmse` : Root Mean Squared Error
- `mae` : Mean Absolute Error
- `max_error` : Max Error
- `median_ae` : Median Absolute Error

#### `Report.available_plots`

```python
Report.available_plots() -> None
```

Print the plot IDs and display names available for the report's detected model
type. The method prints to standard output and does not return a value.

Use these IDs in `build(exclude_plots=[...])`.

Classification plot IDs:

- `confusion_matrix` : Confusion Matrix
- `class_distribution` : Class Distribution
- `predicted_class_distribution` : Predicted Class Distribution
- `per_class_metrics` : Per-Class Metrics

Regression plot IDs:

- `predicted_vs_actual` : Predicted vs Actual
- `residuals` : Residuals vs Predicted
- `residual_hist` : Residual Distribution
- `qq` : Q-Q Plot

#### `Report.add_split`

```python
Report.add_split(
    name: str,
    X: np.ndarray,
    y: np.ndarray,
    y_pred: np.ndarray | None = None,
) -> Report
```

Add one named data split to a standard train/test style report.

##### Parameters

`name` : `str`

- Split identifier, for example `"train"`, `"val"`, or `"test"`.

`X` : `np.ndarray`

- Feature matrix for the split.

`y` : `np.ndarray`

- Ground-truth target values for the split.

`y_pred` : `np.ndarray | None`, default=`None`

- Optional predictions for the split. When omitted, `model.predict(X)` is called.

##### Returns

`Report`

- The same report instance, enabling method chaining.

##### Notes

Call `add_split` once per split that should appear in the report. The split data
is converted to NumPy arrays internally. A report cannot mix `add_split` and
`add_crossval`.


#### `Report.add_crossval`

```python
Report.add_crossval(
    X: np.ndarray,
    y: np.ndarray,
    y_pred: np.ndarray | None = None,
    cv: Any | None = None,
    fold_ids: np.ndarray | None = None,
) -> Report
```

Add a cross-validation prediction set to the report.

##### Parameters

`X` : `np.ndarray`

- Full feature matrix used for cross-validation.

`y` : `np.ndarray`

- Full target vector.

`y_pred` : `np.ndarray | None`, default=`None`

- Optional out-of-fold predictions aligned one-to-one with `y`. When omitted,
  predictions are generated with scikit-learn `cross_val_predict`.

`cv` : `Any | None`, default=`None`

- Optional cross-validation splitter, fold count, or iterable of
  `(train_idx, test_idx)` pairs. Required when `y_pred` is omitted.

`fold_ids` : `np.ndarray | None`, default=`None`

- Optional fold identifier for each row. Cannot be used together with `cv`.
  Useful when `y_pred` was computed elsewhere but fold-level summaries are
  still desired.

##### Returns

`Report`

- The same report instance, enabling method chaining.

##### Notes

Cross-validation reports use one internal split named `"cv"`. When fold IDs are
available, metric values for the `cv` split include fold scores plus summary
statistics: `mean`, `std`, `min`, and `max`. A report cannot mix `add_crossval`
and `add_split`.

#### `Report.add_search`

```python
Report.add_search(search_cv) -> Report
```

Add fitted scikit-learn hyperparameter search results for tuning summaries and
tuning plots.

##### Parameters

`search_cv` : object

- Fitted search object such as `GridSearchCV` or `RandomizedSearchCV`. The
  object must expose a non-empty `cv_results_` mapping.

##### Returns

`Report`

- The same report instance, enabling method chaining.

##### Notes

`mlreport` currently supports tuning visualizations for exactly two searched
hyperparameters. It selects the score column in this order:

1. `mean_test_<refit_metric>` when `search_cv.refit` is a string and the column exists.
2. `mean_test_score` when available.
3. The first sorted `mean_test_*` column.

#### `Report.build`

```python
Report.build(
    exclude_metrics: list[str] | None = None,
    exclude_plots: list[str] | None = None,
) -> Report
```

Compute metrics and generate plots for the data added with `add_split` or
`add_crossval`.

##### Parameters

`exclude_metrics` : `list[str] | None`, default=`None`

- Metric IDs to skip. Use `available_metrics()` to discover valid IDs.

`exclude_plots` : `list[str] | None`, default=`None`

- Plot IDs to skip. Use `available_plots()` to discover valid IDs.

##### Returns

`Report`

- The same report instance, enabling method chaining.

##### Notes

`build()` must be called before any renderer method or `to_dict()`.

#### `Report.to_txt`

```python
Report.to_txt(path: str | None = None) -> Report
```

Render the built report to plain text.

##### Parameters

`path` : `str | None`, default=`None`

- Optional output file path. When omitted, the text report is printed to
  standard output.

##### Returns

`Report`

- The same report instance, enabling method chaining.

##### Notes

Plain-text output excludes plots and tuning plot figures.

#### `Report.to_html`

```python
Report.to_html(path: str) -> Report
```

Render the built report to HTML.

##### Parameters

`path` : `str`

- Output file path for the HTML report.

##### Returns

`Report`

- The same report instance, enabling method chaining.

##### Notes

Plot images are embedded in the HTML as base64 image data.

#### `Report.to_pdf`

```python
Report.to_pdf(path: str) -> Report
```

Render the built report to PDF.

##### Parameters

`path` : `str`

- Output file path for the PDF report.

##### Returns

`Report`

- The same report instance, enabling method chaining.

##### Notes

PDF rendering uses the same report context as HTML, with plots embedded as image
data before PDF generation.

#### `Report.to_json`

```python
Report.to_json(path: str) -> Report
```

Render the built report to JSON.

##### Parameters

`path` : `str`

- Output file path for the JSON report.

##### Returns

`Report`

- The same report instance, enabling method chaining.

##### Notes

JSON output excludes plot figures and tuning plot figures because Matplotlib
figures are not JSON-serializable.

#### `Report.to_md`

```python
Report.to_md(path: str, image_dir: str | None = None) -> Report
```

Render the built report to Markdown.

##### Parameters

`path` : `str`

- Output file path for the Markdown report.

`image_dir` : `str | None`, default=`None`

- Directory where exported plot images are written. Defaults to an `images`
  directory next to `path`.

##### Returns

`Report`

- The same report instance, enabling method chaining.

##### Notes

Markdown output writes plots as PNG files and links to those files from the
generated Markdown. Tuning plots are written with a `tuning_` filename prefix.

#### `Report.to_dict`

```python
Report.to_dict() -> dict
```

Return the built report as a renderer-friendly dictionary.

##### Returns

`dict`

- Report payload containing `meta`, `model`, `data`, `metrics`, `plots`, and
`tuning` sections.
- Example:
  ```python
  {
      "meta": {
          "title": ...,
          "author": ...,
          "description": ...,
          "generated_at": "YYYY-MM-DD HH:MM",
      },
      "model": {
          "name": ...,
          "type": "Classification" | "Regression" | ...,
          "sklearn": "True (v...)" | "False",
          "params": {...},
      },
      "data": {
          "features": ...,
          "splits": {...},
          "total": ...,
          "cv_folds": ...,
          "is_crossval": ...,
          "class_distribution": {...},
          "class_percentages": {...},
      },
      "metrics": {...},
      "plots": {...},
      "tuning": {
          "summary": ...,
          "plots": {...},
      },
  }
  ```


## `mlreport.ComparisonReport`

```python
class mlreport.ComparisonReport(
    reports: list[Report],
    title: str | None = None,
    author: str | None = None,
    description: str | None = None,
    split: str | None = None,
    theme: str = "light",
    cmap: str = "viridis",
)
```

Build and render a comparison report from two or more already-built
[`Report`](#mlreportreport) instances.

The first report in `reports` is treated as the baseline when deltas are shown.
All compared reports must have the same model type and at least one compatible
scalar metric on the selected split.

### Parameters

`reports` : `list[Report]`

- Built `Report` instances to compare. Must contain at least two reports.

`title` : `str | None`, default=`None`

- Optional comparison report title.

`author` : `str | None`, default=`None`

- Optional report author.

`description` : `str | None`, default=`None`

- Optional comparison description.

`split` : `str | None`, default=`None`

- Optional split name to compare across all reports, for example `"test"` or
  `"cv"`. If omitted, `"test"` is preferred when available, otherwise `"cv"` is
  used.

`theme` : `str`, default=`"light"`

- Render theme name. Built-in themes include `"light"` and `"dark"`.

`cmap` : `str`, default=`"viridis"`

- Matplotlib colormap name used for comparison plots.

### Attributes

`reports` : `list[Report]`

- Source reports being compared.

`title` : `str | None`

- Comparison title metadata.

`author` : `str | None`

- Comparison author metadata.

`description` : `str | None`

- Comparison description metadata.

`split` : `str | None`

- Requested comparison split, if supplied.

`theme` : `str`

- Theme used by text, HTML, Markdown, and PDF renderers.

`cmap` : `str`

- Colormap used for generated comparison plots.

### Methods

#### `ComparisonReport.build`

```python
ComparisonReport.build() -> ComparisonReport
```

Validate the source reports and build comparison rows and plots.

##### Returns

`ComparisonReport`

- The same comparison instance, enabling method chaining.

##### Notes

`build()` calls `to_dict()` on each source report, so every source report must
already be built. If duplicate model class names appear, the comparison assigns
unique display keys such as `"RandomForestClassifier (1)"` and
`"RandomForestClassifier (2)"`.

For classification comparisons, generated comparison plot groups currently
include:

- `confusion_matrix`
- `per_class_metrics`

For regression comparisons, generated comparison plot groups currently include:

- `predicted_vs_actual`
- `residual_hist`

#### `ComparisonReport.to_txt`

```python
ComparisonReport.to_txt(path: str | None = None) -> ComparisonReport
```

Render the built comparison report to plain text.

##### Parameters

`path` : `str | None`, default=`None`

- Optional output file path. When omitted, the text report is printed to
  standard output.

##### Returns

`ComparisonReport`

- The same comparison instance, enabling method chaining.

#### `ComparisonReport.to_html`

```python
ComparisonReport.to_html(path: str) -> ComparisonReport
```

Render the built comparison report to HTML.

##### Parameters

`path` : `str`

- Output file path for the HTML comparison report.

##### Returns

`ComparisonReport`

- The same comparison instance, enabling method chaining.

##### Notes

Comparison plots are embedded in the HTML as base64 image data.

#### `ComparisonReport.to_pdf`

```python
ComparisonReport.to_pdf(path: str) -> ComparisonReport
```

Render the built comparison report to PDF.

##### Parameters

`path` : `str`

- Output file path for the PDF comparison report.

##### Returns

`ComparisonReport`

- The same comparison instance, enabling method chaining.

#### `ComparisonReport.to_json`

```python
ComparisonReport.to_json(path: str) -> ComparisonReport
```

Render the built comparison report to JSON.

##### Parameters

`path` : `str`

- Output file path for the JSON comparison report.

##### Returns

`ComparisonReport`

- The same comparison instance, enabling method chaining.

#### `ComparisonReport.to_md`

```python
ComparisonReport.to_md(path: str, image_dir: str | None = None) -> ComparisonReport
```

Render the built comparison report to Markdown.

##### Parameters

`path` : `str`

- Output file path for the Markdown comparison report.

`image_dir` : `str | None`, default=`None`

- Directory where exported comparison plot images are written. Defaults to an
  `images` directory next to `path`.

##### Returns

`ComparisonReport`

- The same comparison instance, enabling method chaining.

##### Notes

Markdown output writes comparison plot images as PNG files and links to those
files from the generated Markdown.

#### `ComparisonReport.to_dict`

```python
ComparisonReport.to_dict() -> dict
```

Return the built comparison report as a renderer-friendly dictionary.

##### Returns

`dict`

- Comparison payload containing `meta`, `comparison`, `models`, and `metrics`
sections.
- Example:
  ```python
  {
      "meta": {
          "title": ...,
          "author": ...,
          "description": ...,
          "generated_at": "YYYY-MM-DD HH:MM",
      },
      "comparison": {
          "baseline_key": ...,
          "split": ...,
          "mixed_splits": ...,
      },
      "models": [
          {
              "index": ...,
              "key": ...,
              "title_name": ...,
              "description": ...,
              "name": ...,
              "type": ...,
              "comparison_split": ...,
              "data_label": ...,
              "tuned_label": ...,
              "params": {...},
              "param_count": ...,
              "is_baseline": ...,
          },
      ],
      "metrics": [
          {
              "metric_id": ...,
              "metric_name": ...,
              "direction": "max" | "min",
              "keys": [...],
              "values": {...},
              "splits": {...},
              "deltas": {...},
              "best_key": ...,
              "best_index": ...,
          },
      ],
  }
  ```


# Classification Example: Iris


## Classification Setup


In [ ]:
from pathlib import Path

from sklearn.datasets import load_iris
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from mlreport import ComparisonReport, Report

Path("reports").mkdir(exist_ok=True)
X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
author = "Lucas Summers"
theme = "light"


## Appendix A: Classification Train/Test Split


In [ ]:
split_model = GaussianNB()

split_model.fit(X_train, y_train)
y_pred_train = split_model.predict(X_train)
y_pred_test = split_model.predict(X_test)

split_report = Report(
    split_model,
    title="Appendix A: Iris Train/Test Split Report",
    author=author,
    description="Gaussian Naive Bayes evaluated on a standard train/test split.",
    theme=theme,
)

split_report.add_split("train", X_train, y_train, y_pred_train)
split_report.add_split("test", X_test, y_test, y_pred_test)
split_report.build().to_pdf("reports/iris_split_report.pdf").to_txt()


## Appendix B: Classification Cross-Validation


In [ ]:
cv_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", C=2.0, gamma="scale")),
    ]
)

cv_model.fit(X_train, y_train)

cv_report = Report(
    cv_model,
    title="Appendix B: Iris Cross-Validation Report",
    author=author,
    description="Support Vector Classifier evaluated with 5-fold stratified cross-validation.",
    theme=theme,
)

cv_report.add_crossval(X_train, y_train, cv=cv)
cv_report.build().to_pdf("reports/iris_cv_report.pdf").to_txt()


## Appendix C: Classification Tuned Cross-Validation


In [ ]:
search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid={
        "n_estimators": [50, 100, 150],
        "learning_rate": [0.03, 0.1, 0.2],
    },
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

search.fit(X_train, y_train)
best_tuned_model = search.best_estimator_

tuned_cv_report = Report(
    best_tuned_model,
    title="Appendix C: Iris Tuned Cross-Validation Report",
    author=author,
    description="Gradient Boosting tuned with GridSearchCV and evaluated with 5-fold CV.",
    theme=theme,
)

tuned_cv_report.add_crossval(X_train, y_train, cv=cv)
tuned_cv_report.add_search(search)
tuned_cv_report.build().to_pdf("reports/iris_tuned_cv_report.pdf").to_txt()


## Appendix D: Classification Comparison


In [ ]:
comparison = ComparisonReport(
    reports=[
        split_report,
        cv_report,
        tuned_cv_report,
    ],
    title="Appendix D: Iris Comparison Report",
    author=author,
    description="Comparison of three classification workflows on the Iris dataset.",
    theme=theme,
)

comparison.build().to_pdf("reports/iris_comparison_report.pdf", include_model_reports=False).to_txt()


# Regression Example: Diabetes


## Regression Setup


In [ ]:
from pathlib import Path

from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from mlreport import ComparisonReport, Report

Path("reports").mkdir(exist_ok=True)

X, y = load_diabetes(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)
author = "Lucas Summers"
theme = "light"


## Appendix E: Regression Train/Test Split


In [ ]:
split_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ]
)

split_model.fit(X_train, y_train)
y_pred_train = split_model.predict(X_train)
y_pred_test = split_model.predict(X_test)

split_report = Report(
    split_model,
    title="Appendix E: Diabetes Train/Test Split Report",
    author=author,
    description="Linear Regression evaluated on a standard train/test split.",
    theme=theme,
)

split_report.add_split("train", X_train, y_train, y_pred_train)
split_report.add_split("test", X_test, y_test, y_pred_test)
split_report.build().to_pdf("reports/diabetes_split_report.pdf").to_txt()


## Appendix F: Regression Cross-Validation


In [ ]:
cv_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=6,
    random_state=42,
)
cv_model.fit(X_train, y_train)

cv_report = Report(
    cv_model,
    title="Appendix F: Diabetes Cross-Validation Report",
    author=author,
    description="Random Forest evaluated with 5-fold cross-validation.",
    theme=theme,
)

cv_report.add_crossval(X_train, y_train, cv=cv)
cv_report.build().to_pdf("reports/diabetes_cv_report.pdf").to_txt()


## Appendix G: Regression Tuned Cross-Validation


In [ ]:
search = GridSearchCV(
    estimator=Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", KNeighborsRegressor()),
        ]
    ),
    param_grid={
        "model__n_neighbors": [3, 5, 7, 9],
        "model__weights": ["uniform", "distance"],
    },
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)

search.fit(X_train, y_train)
best_tuned_model = search.best_estimator_

tuned_cv_report = Report(
    best_tuned_model,
    title="Appendix G: Diabetes Tuned Cross-Validation Report",
    author=author,
    description="KNN regressor tuned with GridSearchCV and evaluated with 5-fold CV.",
    theme=theme,
)

tuned_cv_report.add_crossval(X_train, y_train, cv=cv)
tuned_cv_report.add_search(search)
tuned_cv_report.build().to_pdf("reports/diabetes_tuned_cv_report.pdf").to_txt()


## Appendix H: Regression Comparison


In [ ]:
comparison = ComparisonReport(
    reports=[
        split_report,
        cv_report,
        tuned_cv_report,
    ],
    title="Appendix H: Diabetes Comparison Report",
    author=author,
    description="Comparison of three regression workflows on the Diabetes dataset.",
    theme=theme,
)

comparison.build().to_pdf("reports/diabetes_comparison_report.pdf", include_model_reports=False).to_txt()
